<div align="center">

# Lista 5

</div>

<div align="center">

## Konfiguracja

</div>

In [28]:
CHROMOSOME_LEN = 100
POPULATION_LEN = 20
ITERATION_NUM = 10000
LOG_ITER = 10

<div align="center">

## Helper Functions

</div>

In [29]:
import numpy as np

# funkcje pomocnicze

<div align="center">

## Population Based Incremental Learning (PBIL)

</div>

In [30]:
import numpy as np
import random
from typing import List

def binary_random(p):
    return 1 if random.random() <= p else 0

def random_individual(p: List[float]) -> np.ndarray:
    return np.array([binary_random(pi) for pi in p])

def random_population(p: List[float], N: int = POPULATION_LEN) -> np.ndarray:
    return np.array([random_individual(p) for _ in range(N)])

def population_evaluation(P: List[List[int]], F):
    return np.array([F(x) for x in P])

def population_based_incremental_learning(F, N: int, O1, O2, O3):
    log_best_values = []
    log_best_individuals = []
    log_prob_vector = []

    prob_vector = np.array([0.5 for _ in range(N)])                      # Initial Probabilty Vector
    ran_population = random_population(prob_vector, POPULATION_LEN)      # Initial Random Population 
    fitness = population_evaluation(ran_population, F)
    for iter in range(ITERATION_NUM):
        x = ran_population[np.argmax(fitness)]

        for i in range(N):
            prob_vector[i] = prob_vector[i] * (1 - O1) + (x[i] * O1)

        for i in range(N):
            if random.random() <= O2:
                prob_vector[i] = prob_vector[i] * (1 - O3) + O3 * binary_random(0.5)

        ran_population = random_population(prob_vector, POPULATION_LEN)
        fitness = population_evaluation(ran_population, F)

        if iter % LOG_ITER == 0:
            log_best_values.append(int(max(fitness)))
            log_best_individuals.append(ran_population[np.argmax(fitness)].copy())
            log_prob_vector.append(prob_vector.copy())

    return log_best_values, log_best_individuals, log_prob_vector

<div align="center">

## Compact Genetic Algorithm (CGA)

</div>

In [31]:
import numpy as np

def individual_evaluation(x: List[int], F):
    return F(x)

def compact_genetic_algorithm(F, N: int, O):
    log_best_values = []
    log_best_individuals = []
    log_prob_vector = []

    prob_vector = np.array([0.5 for _ in range(N)])
    x1 = random_individual(prob_vector)
    x2 = random_individual(prob_vector)
    v1 = individual_evaluation(x1, F)
    v2 = individual_evaluation(x2, F)
    for iter in range(ITERATION_NUM):
        if v1 > v2:
            best_individual = x1
            worst_individual = x2
        else:
            best_individual = x2
            worst_individual = x1

        for i in range(N):
            if best_individual[i] == 1 and worst_individual[i] == 0:
                prob_vector[i] += O
            elif best_individual[i] == 0 and worst_individual[i] == 1:
                prob_vector[i] -= O
            
        x1 = random_individual(prob_vector)
        x2 = random_individual(prob_vector)
        v1 = individual_evaluation(x1, F)
        v2 = individual_evaluation(x2, F)

        if iter % LOG_ITER == 0:
            log_best_values.append(int(v1 if v1 > v2 else v2))
            log_best_individuals.append(x1 if v1 > v2 else x2)
            log_prob_vector.append(prob_vector.copy())
        
    return log_best_values, log_best_individuals, log_prob_vector, 

<div align="center">

## Univariate Marginal Distribution Algorithm (UMDA)

</div>

In [32]:
def select_best(P: List[List[int]], fitness, N, M):
    assert(N >= M, 'Nie można wybrać więcej osobników niż jest w populacji')
    P = np.array(P) 
    idx = np.argsort(fitness)[::-1]
    return P[idx[:M]]

def univariate_marginal_distribution_algorithm(F, N):
    log_best_values = []
    log_best_individuals = []
    log_prob_vector = []

    prob_vector = np.array([0.5 for _ in range(N)])
    rand_pop = random_population(prob_vector, POPULATION_LEN)
    fitness = population_evaluation(rand_pop, F)
    for iter in range(ITERATION_NUM):
        Ps = select_best(rand_pop, fitness, POPULATION_LEN, POPULATION_LEN // 2)
        prob_vector = np.mean(Ps, axis=0)
        rand_pop = random_population(prob_vector, POPULATION_LEN)
        fitness = population_evaluation(rand_pop, F)

        if iter % LOG_ITER == 0:
            best_idx = np.argmax(fitness)
            log_best_values.append(int(fitness[best_idx]))
            log_best_individuals.append(rand_pop[np.argmax(fitness)].copy())
            log_prob_vector.append(prob_vector.copy())

    return log_best_values, log_best_individuals, log_prob_vector


<>:2: SyntaxWarning:

assertion is always true, perhaps remove parentheses?

<>:2: SyntaxWarning:

assertion is always true, perhaps remove parentheses?

C:\Users\piotr\AppData\Local\Temp\ipykernel_31716\2415636336.py:2: SyntaxWarning:

assertion is always true, perhaps remove parentheses?



<div align="center">

## Mutual Information Maximization for Input Clustering (MIMIC)

</div>

In [33]:
def mutual_information_maximization_for_input_clustering():
    pass

<div align="center">

## (NSGA-II)

</div>

<div align="center">

## One Max and Deceptive One Max

</div>

In [34]:
import numpy as np
from typing import List

def one_max(seq: List[int]) -> int:
    return np.sum(seq) 

def deceptive_one_max(seq: List[int]) -> int:
    res = np.sum(seq)
    n = len(seq)
    if res == n:
        return n  
    else:
        return n - res - 1

In [35]:
import plotly.express as px
import pandas as pd
import numpy as np

def plot_ea_results(results: dict, title: str = "Porównanie algorytmów ewolucyjnych"): 
    # Budujemy dataframe: algorytm, iteracja, wartość
    df_list = []
    for name, values in results.items():
        df_list.append(
            pd.DataFrame({
                "algorithm": name,
                "iter": np.arange(len(values)) * LOG_ITER,
                "best_values": values
            })
        )
    df = pd.concat(df_list, ignore_index=True)
    
    # Wykres
    fig = px.line(
        df,
        x="iter",
        y="best_values",
        color="algorithm",
        title=title
    )
    
    # Czarne tło i białe linie
    fig.update_layout(
        plot_bgcolor="black",
        paper_bgcolor="black",
        font_color="white",
        title_font_color="white",
        legend_title_font_color="white",
        xaxis_title="Iteracja",
        yaxis_title="Najlepsze wartości"
    )
    
    fig.update_traces(
        line=dict(width=3),
    )
    
    # Białe osie
    fig.update_xaxes(color="white", gridcolor="gray")
    fig.update_yaxes(color="white", gridcolor="gray")
    
    fig.show()


<div align="center">

## Zadanie 1

</div>

In [36]:
best_values_pbil, best_individuals_pbil, prob_vectors_pbil = population_based_incremental_learning(
    one_max, CHROMOSOME_LEN, 0.2, 0.1, 0.1
)
best_values_cga, best_individuals_cga, prob_vectors_cga = compact_genetic_algorithm(
    one_max, CHROMOSOME_LEN, 0.2
)
best_values_umda, best_individuals_umda, prob_vectors_umda = univariate_marginal_distribution_algorithm(
    one_max, CHROMOSOME_LEN
)

plot_ea_results(
    {
        "UMDA": best_values_umda,
        "PBIL": best_values_pbil,
        "cGA": best_values_cga,
        # "MIMIC": best_values_mimic,
        # "ngsa-II": 

    },
    title="Porównanie PBIL i cGA na problemie OneMax"
)

print(best_individuals_pbil[-1:-5:-1])
print(best_individuals_cga[-1:-5:-1])
print(best_individuals_umda[-1:-5:-1])


[array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1,

<div align="center">

## Zadanie 2

</div>